# Pinecone Vector Search — Sample Index

A minimal, end-to-end notebook for creating a **Pinecone serverless index**, embedding sample documents with a local `sentence-transformers` model, upserting vectors, and running similarity queries (including metadata filters).

**What this notebook does**
1. Installs dependencies
2. Authenticates to Pinecone via API key (prompted at runtime — never stored in the notebook)
3. Creates a serverless index (skips creation if it already exists)
4. Generates embeddings for sample documents using `all-MiniLM-L6-v2` (384-dim, ~80 MB)
5. Upserts vectors with metadata
6. Runs a semantic query and a filtered query
7. Provides cleanup commands

**Why this flow?** It keeps the control plane (index lifecycle) and the data plane (upsert/query) visually separated, which maps to how you'd wire this up in a real pipeline.

## 1. Install dependencies

- `pinecone` — official Pinecone SDK (v5+, serverless-native)
- `sentence-transformers` — for generating embeddings locally (no extra API keys)
- `torch` — backend for `sentence-transformers` (installed transitively if not present)

In [ ]:
# %pip install --quiet pinecone sentence-transformers

## 2. Imports

In [1]:
import os
import time
from getpass import getpass

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

## 3. Authenticate

Prefer an environment variable (`PINECONE_API_KEY`). If it's not set, we fall back to a `getpass` prompt so the key never lands in the notebook cell output or in version control.

> Grab a key from https://app.pinecone.io → *API Keys*.

In [2]:
api_key = os.getenv("PINECONE_API_KEY") or getpass("Enter your Pinecone API key: ")
pc = Pinecone(api_key=api_key)

# Sanity check — lists existing indexes in the project
existing = [idx["name"] for idx in pc.list_indexes()]
print(f"Existing indexes in project: {existing}")

Existing indexes in project: ['sample-movies']


## 4. Create a serverless index

Pinecone serverless indexes scale to zero and bill by usage. Key choices:

| Parameter | Value | Rationale |
|---|---|---|
| `dimension` | `384` | Matches `all-MiniLM-L6-v2` output |
| `metric` | `cosine` | Standard for normalized sentence embeddings |
| `cloud` / `region` | `aws` / `us-east-1` | Free-tier compatible default |

If the index already exists, we skip creation and reuse it — idempotent behavior is important for any pipeline you'd run more than once.

In [3]:
INDEX_NAME = "sample-vector-index"
DIMENSION = 384
METRIC = "cosine"

if INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # Wait until the index is ready before upserting
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"Created index '{INDEX_NAME}'.")
else:
    print(f"Index '{INDEX_NAME}' already exists — reusing it.")

pc.describe_index(INDEX_NAME)

Created index 'sample-vector-index'.


{
    "name": "sample-vector-index",
    "metric": "cosine",
    "host": "sample-vector-index-23f8m0l.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api

In [4]:
# Get a handle to the index for data-plane operations
index = pc.Index(INDEX_NAME)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 04:47:12 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '1',
                                    'x-pinecone-request-latency-ms': '1',
                                    'x-pinecone-response-duration-ms': '2'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

## 5. Prepare sample data

Ten short documents across three topical clusters (data platforms, ML infra, sports) so we can visually confirm the similarity search is doing something meaningful.

In [5]:
documents = [
    {"id": "doc-01", "text": "Databricks Unity Catalog provides centralized governance for data and AI assets.",         "category": "data-platform"},
    {"id": "doc-02", "text": "Snowflake Horizon offers a unified governance layer across databases and applications.",  "category": "data-platform"},
    {"id": "doc-03", "text": "Delta Lake brings ACID transactions and time travel to cloud object storage.",            "category": "data-platform"},
    {"id": "doc-04", "text": "Apache Iceberg is an open table format designed for huge analytic datasets.",             "category": "data-platform"},
    {"id": "doc-05", "text": "Vector databases store high-dimensional embeddings for semantic search.",                 "category": "ml-infra"},
    {"id": "doc-06", "text": "Retrieval-augmented generation grounds language model responses in external documents.", "category": "ml-infra"},
    {"id": "doc-07", "text": "Fine-tuning adapts a pretrained foundation model to a specific downstream task.",         "category": "ml-infra"},
    {"id": "doc-08", "text": "Liverpool won the Premier League after a dominant second-half run.",                      "category": "sports"},
    {"id": "doc-09", "text": "The cricket match ended in a draw after rain interrupted play on day four.",              "category": "sports"},
    {"id": "doc-10", "text": "Formula 1 teams unveiled new aerodynamic packages for the upcoming season.",              "category": "sports"},
]

print(f"Prepared {len(documents)} documents.")

Prepared 10 documents.


## 6. Generate embeddings

`all-MiniLM-L6-v2` is a 22M-parameter sentence encoder that outputs L2-normalized 384-dim vectors — good enough for a demo, cheap on CPU. For production you'd swap in something like `BAAI/bge-large-en-v1.5` or a hosted embedding API.

In [6]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [d["text"] for d in documents]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
print(f"Generated embeddings with shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generated embeddings with shape: (10, 384)


## 7. Upsert vectors

Each vector carries an `id`, the embedding, and a metadata payload. Metadata is queryable via Pinecone's filter language (used in the next section). Pinecone recommends batching upserts — at this scale one call is fine, but the loop pattern below generalizes to larger datasets.

In [7]:
vectors = [
    {
        "id": doc["id"],
        "values": emb.tolist(),
        "metadata": {"text": doc["text"], "category": doc["category"]},
    }
    for doc, emb in zip(documents, embeddings)
]

# Batch in chunks of 100 (Pinecone's recommended upsert size)
BATCH_SIZE = 100
for i in range(0, len(vectors), BATCH_SIZE):
    index.upsert(vectors=vectors[i : i + BATCH_SIZE])

print(f"Upserted {len(vectors)} vectors.")

Upserted 10 vectors.


In [8]:
# Upserts are asynchronous server-side — give the index a moment, then check
time.sleep(2)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '183',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 04:48:36 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}

## 8. Query the index

### 8a. Plain semantic search

In [9]:
query_text = "How do I govern data assets on a lakehouse?"
query_vec = model.encode([query_text], normalize_embeddings=True)[0].tolist()

results = index.query(
    vector=query_vec,
    top_k=3,
    include_metadata=True,
)

print(f"Query: {query_text}\n")
for match in results["matches"]:
    print(f"  [{match['score']:.4f}]  {match['id']}  ({match['metadata']['category']})")
    print(f"           {match['metadata']['text']}\n")

Query: How do I govern data assets on a lakehouse?

  [0.4918]  doc-01  (data-platform)
           Databricks Unity Catalog provides centralized governance for data and AI assets.

  [0.3996]  doc-03  (data-platform)
           Delta Lake brings ACID transactions and time travel to cloud object storage.

  [0.3501]  doc-02  (data-platform)
           Snowflake Horizon offers a unified governance layer across databases and applications.



### 8b. Semantic search with a metadata filter

Same query, but restricted to the `ml-infra` category. Filters are evaluated server-side before the ANN search, so there's no wasted work.

In [10]:
query_text = "What technique grounds LLMs in external knowledge?"
query_vec = model.encode([query_text], normalize_embeddings=True)[0].tolist()

results = index.query(
    vector=query_vec,
    top_k=3,
    include_metadata=True,
    filter={"category": {"$eq": "ml-infra"}},
)

print(f"Query (filtered to ml-infra): {query_text}\n")
for match in results["matches"]:
    print(f"  [{match['score']:.4f}]  {match['id']}")
    print(f"           {match['metadata']['text']}\n")

Query (filtered to ml-infra): What technique grounds LLMs in external knowledge?

  [0.2064]  doc-05
           Vector databases store high-dimensional embeddings for semantic search.

  [0.2036]  doc-07
           Fine-tuning adapts a pretrained foundation model to a specific downstream task.

  [0.2488]  doc-06
           Retrieval-augmented generation grounds language model responses in external documents.



## 9. Fetch and delete individual vectors

Useful for building deletion pipelines — e.g. when a source document is removed, you delete by ID and the semantic search immediately reflects the change.

In [11]:
# Fetch by IDs
fetched = index.fetch(ids=["doc-01", "doc-05"])
for vec_id, vec in fetched.vectors.items():
    print(f"{vec_id}: {vec.metadata['text']}")

doc-01: Databricks Unity Catalog provides centralized governance for data and AI assets.
doc-05: Vector databases store high-dimensional embeddings for semantic search.


In [12]:
# Delete a specific vector
index.delete(ids=["doc-10"])
time.sleep(1)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '181',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 04:49:10 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '5',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 9}},
 'storageFullness': 0.0,
 'total_vector_count': 9,
 'vector_type': 'dense'}

## 10. Cleanup (optional)

Serverless indexes bill by storage and read/write units — delete the index when you're done experimenting.

> Uncomment the line below only when you actually want to tear it down.

In [13]:
pc.delete_index(INDEX_NAME)
print(f"Deleted index '{INDEX_NAME}'.")

Deleted index 'sample-vector-index'.
